# Mantis Vision — Train in Google Colab

Run these cells top to bottom. Before starting: **Runtime -> Change runtime type -> GPU**.

This notebook: clones the repo, installs dependencies, lets you upload your labeled photo dataset directly, trains the EfficientNet-B0 health classifier, evaluates it, and lets you download the resulting checkpoint.

## 1. Clone the repo

In [ ]:
!git clone https://github.com/Arran0/MantisVision.git
%cd MantisVision/ml

## 2. Install dependencies

Colab already has torch/torchvision preinstalled with GPU support, so this just adds the extra packages the pipeline needs.

In [ ]:
!pip install -q grad-cam fastapi python-multipart

## 3. Upload your dataset

On your own computer, organize your photos into `train/validation/test/<Class>/` folders (see `docs/DATASET_LABELING_GUIDE.md`), then **zip the contents** of that top folder — select `train`, `validation`, and `test` themselves and zip those three, not their parent folder (zipping the parent creates an extra wrapper level around them, which the cell below can auto-fix if it happens anyway).

Run the cell below, then use the file picker to select that `.zip`.

In [ ]:
import pathlib
import shutil
import zipfile

from google.colab import files

dataset_dir = pathlib.Path("dataset")
dataset_dir.mkdir(parents=True, exist_ok=True)

print("Select the zip file containing your train/validation/test folders...")
uploaded = files.upload()

for name in uploaded:
    if name.lower().endswith(".zip"):
        print(f"Extracting {name} ...")
        with zipfile.ZipFile(name) as zf:
            zf.extractall(dataset_dir)
    else:
        print(f"Skipping non-zip file: {name} (expected a single .zip upload)")

# Auto-fix: flatten one level of wrapper folder if train/validation/test ended
# up nested inside it (happens if you zipped the parent folder instead of its
# contents).
expected_splits = {"train", "validation", "test"}
top_level_dirs = {p.name for p in dataset_dir.iterdir() if p.is_dir()}
if not expected_splits.issubset(top_level_dirs):
    for child in dataset_dir.iterdir():
        if not child.is_dir():
            continue
        grandchildren = {p.name for p in child.iterdir() if p.is_dir()}
        if expected_splits.issubset(grandchildren):
            print(f"Found train/validation/test nested inside '{child.name}/', flattening...")
            for item in child.iterdir():
                shutil.move(str(item), str(dataset_dir / item.name))
            child.rmdir()
            break


def print_tree(path, prefix="", depth=0, max_depth=3):
    if depth > max_depth:
        return
    for entry in sorted(path.iterdir()):
        print(f"{prefix}{entry.name}{'/' if entry.is_dir() else ''}")
        if entry.is_dir():
            print_tree(entry, prefix + "  ", depth + 1, max_depth)


print(f"\nContents of {dataset_dir.resolve()}:\n")
print_tree(dataset_dir)

## 4. Validate the dataset

Checks every class folder exists, every image opens correctly, and flags underrepresented classes.

In [ ]:
!python -m src.data.validate_dataset

## 5. Train

Two-phase schedule (frozen backbone, then fine-tune) with early stopping. Saves the best checkpoint to `ml/checkpoints/best_model.pt`. This can take a while depending on dataset size — Colab's free GPU tier helps a lot here.

In [ ]:
!python -m src.train

## 6. Evaluate

Accuracy/precision/recall/F1 (macro + per-class), confusion matrix, ROC AUC on the held-out test split.

In [ ]:
!python -m src.evaluate

from IPython.display import Image, display
display(Image(filename="reports/confusion_matrix.png"))

## 7. Download the trained checkpoint

Colab runtimes are ephemeral — grab the checkpoint now so you don't lose it when the session disconnects. You'll need this file to run the inference API later.

In [ ]:
from google.colab import files
files.download("checkpoints/best_model.pt")

## Optional: try a Grad-CAM explanation on one photo

Upload a single test photo and see the heatmap of what the model looked at.

In [ ]:
from google.colab import files
uploaded = files.upload()  # pick one photo
photo_path = list(uploaded.keys())[0]
!python -m src.gradcam "{photo_path}"

from IPython.display import Image, display
import pathlib
display(Image(filename=str(pathlib.Path(photo_path).with_suffix('.gradcam.png'))))